# PyTorch Cheatsheet

[Pytorch Documentation](https://pytorch.org/docs/stable/index.html)

In [ ]:
# !python -m pip install torch==2.0.0 torchvision torchaudio

In [ ]:
import torch


### GPUs

PyTorch tensors have inherent GPU support. Specifying to use the GPU memory and CUDA cores for storing and performing tensor calculations is easy; the cuda package can help determine whether GPUs are available, and the package's cuda() method assigns a tensor to the GPU.


In [ ]:
# Checking whether GPU is available
torch.cuda.is_available()

# Move to GPU # 
# Note: if you did not install PyTorch with CUDA, this will throw an error
# Note 2: if your laptop does not have an Nvidia GPU, you don't need to install the CUDA-enabled version of PyTorch
t.cuda()

## Specifying a neural network

**Step 1:** define your network as a subclass of either the nn.Module or a subclass of this module

**Step 2:** specify the architecture of your network (size and number of layers)
- Define the init function

**Step 3:** say how the layers connect (for each layer, specify input size, output size and activation function)
- Define the forward function


In [ ]:
from torch import tensor
from torch import nn
from torbch import sigmoid
import torch.nn.functional as F
import torch.optim as optim

#### MLP Example: Step 1 and 2
- Implementing a multi-layer perceptron with two layers.
- The input is a text so the size of the input layer is the maximum length of the texts in our dataset.
- The size of the output layer is the number of classes the classifier is handling.

In [ ]:
# Our MultilayerPerceptron is a subclass of the nn.Module Class
class MultilayerPerceptron(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MultilayerPerceptron, self).__init__()
        # The network consists of two fully connected layers 
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)

#### MLP Example: Step 3
- Connecting the layers

In [ ]:
# Specifying how the layers connect
def forward(self, x_in, apply_softmax=False):
        """The forward pass of the MLP
        Args:
            x_in: an input data tensor 
            apply_softmax=false because we use the cross-entropy loss
        Returns:
            the resulting tensor
        """
        # Apply non linear Relu activation function to the output of the first fully connected layer
        intermediate = F.relu(self.fc1(x_in))
        # Apply non linear Relu activation function to the output of the second fully connected layer   
        # No need for softmax function here 
        # it is already included in the CrossEntropyLoss computation
        output = self.fc2(intermediate)
        return output

### Embedding layer

The embedding layer is used to transform our sparse one-hot vector (sparse as most of the elements are 0) into a dense embedding vector (dense as the dimensionality is a lot smaller and all the elements are real numbers). This embedding layer is simply a single fully connected layer. 

The nn.Embedding module holds a Tensor of dimension (vocab_size, embedding_size), i.e. of the size of the vocabulary x the dimension of each vector embedding, and a method for retrieving the embedding of a word. 


In [ ]:
# Create an embedding layer
# Parameters: (vocab_size, embedding_size)
embedding = nn.Embedding(1000,128)
# Print out the embedding of the token represented by index 3
embedding(torch.LongTensor([3]))

### Pooling
The output of a layer can be aggregated using mean/max/min pooling.

In [ ]:
# Define a simple MLP for binary sentiment classification
class MultilayerPerceptron(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        ....

    def forward(self, input_ids):
        embedded = self.embedding(input_ids) # Shape: (Batch Size, Sentence Length, Embedding Dimension)
        pooled = embedded.mean(dim=1)       # Average along the sentence length dimension
        x = F.relu(self.fc1(pooled))      # Apply ReLU activation after first linear layer
        x = torch.sigmoid(self.fc2(x))   # Apply sigmoid to output a probability (0 to 1)
        return x              # Return predicted probability

#### Creating a model

In [ ]:
# Size of the input layer (max length of the input text)
input_size = 471
# Size of the hidden layers
hidden_size = 128
# Size of the output layer (classification layer)
num_classes = 5
mlp = MultilayerPerceptron(input_size, hidden_size, num_classes)

### Loading data into tensors

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
# the TensorDataset is a ready to use class to represent your data as list of tensors. 
# Note that input_features and labels must match on the length of the first dimension
# (Same number of instances and of labels)
train_set = TensorDataset(X_train, Y_train)
valid_set = TensorDataset(X_valid, Y_valid)
# DataLoader shuffles, batches and loads  the data in parallel using multiprocessing workers
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)

### Training 

In [ ]:
### Training

def fit(model, epochs):
    # Specify the loss function  and the optimizer
    criterion = nn.CrossEntropyLoss()  
    optimizer = optim.Adam(model.parameters()) 
    # Iterate over epochs (i.e., slices of the data)
    for epoch in range(epochs):
        model.train()                # Set the module in training mode 
        total_loss = num = 0         # Initialise the loss to 0
        # Iterate over batches of (x,y) pairs in the training data
        for x, y in train_loader:
            optimizer.zero_grad()    # null the gradients 
            y_scores = model(x)      # predict labels for the batch
            loss = criterion(y_scores, y)  # calculate the loss
            loss.backward()          # Back propagate
            optimizer.step()         # Adjust the weights
       print(epoch, *perf(model, train_loader))

# Train for 10 epochsb
fit(mlp,10)

### Inference

In [ ]:
from sklearn.metrics import confusion_matrix

def predict(model, loader):
    # No drop out
    model.eval()
    correct = 0
    gold = outputs = []
    for x, y in loader:
    # No gradient computation, weights remain unchanged
    with torch.no_grad():
        # Compute the scores for the instances in the input batch
        y_scores = model(x)
        # Compute the predictions
        # y_scores = matrix    
        # max(y_scores,1) = max value on lines    
        # max(y_scores,1)[1] = index of max value on lines    
        y_preds = torch.max(y_scores, 1)[1]
        # Store the gold value
        gold = gold+y.tolist()
        # Store the predicted value
        outputs = outputs+y_preds.tolist()
    print(confusion_matrix(gold,outputs))
predict(rnn_model,valid_loader)

### Evaluating

In [ ]:
def perf(model, loader):
# define the loss
    criterion = nn.CrossEntropyLoss()
# No drop out
    model.eval()
    total_loss = correct = num = 0
    for x, y in loader:
# No gradient computation, weights remain unchanged
      with torch.no_grad():
# Compute the scores for the instances in the input batch
        y_scores = model(x)
# Compute the loss
        loss = criterion(y_scores, y)
# Compute the predictions
        y_pred = torch.max(y_scores, 1)[1]
# Update the batch loss
        total_loss += loss.item()
        num += len(y)
    return total_loss / num